# Querying Unsupported Topics in RAG

Demonstration: What happens when a question is asked about content **not present** in the document corpus?

In [0]:
from databricks.sdk import WorkspaceClient
import pandas as pd

# Initialize client
w = WorkspaceClient()

# The UNSUPPORTED question - not covered by any HR document
unsupported_question = "What is the company's stock option vesting schedule?"

print("🔍 QUERYING UNSUPPORTED TOPIC")
print("=" * 100)
print(f"Question: {unsupported_question}")
print("=" * 100)

# First, let's see what topics ARE available in our corpus
print("\n📚 Available Topics in HR Document Corpus:")
available_topics = spark.sql("""
    SELECT DISTINCT topic
    FROM hr_catalog.hr_core.hr_document_chunks
    ORDER BY topic
""").collect()

for row in available_topics:
    print(f"  ✓ {row['topic']}")

print("\n❌ Notice: 'Stock Options', 'Compensation', or 'Equity' topics are NOT in the corpus\n")

# Simulate keyword-based search (since vector search has schema issues)
print("\n" + "=" * 100)
print("SIMULATED SEARCH RESULTS - Best Keyword Matches (Top 5)")
print("=" * 100)
print("\nSearching for: 'stock', 'option', 'vesting', 'equity', 'compensation'...")

try:
    # Search for any chunks containing stock-related keywords
    keyword_results = spark.sql("""
        SELECT 
            document_name,
            topic,
            chunk_text,
            chunk_id,
            CASE 
                WHEN LOWER(chunk_text) LIKE '%stock%' THEN 1
                WHEN LOWER(chunk_text) LIKE '%option%' THEN 1
                WHEN LOWER(chunk_text) LIKE '%vesting%' THEN 1
                WHEN LOWER(chunk_text) LIKE '%equity%' THEN 1
                WHEN LOWER(chunk_text) LIKE '%compensation%' THEN 1
                ELSE 0
            END as relevance_score
        FROM hr_catalog.hr_core.hr_document_chunks
        WHERE 
            LOWER(chunk_text) LIKE '%stock%' 
            OR LOWER(chunk_text) LIKE '%option%'
            OR LOWER(chunk_text) LIKE '%vesting%'
            OR LOWER(chunk_text) LIKE '%equity%'
            OR LOWER(chunk_text) LIKE '%compensation%'
        ORDER BY relevance_score DESC
        LIMIT 5
    """).collect()
    
    if keyword_results:
        print("\n📊 Keyword Search Found Matches:")
        print("-" * 100)
        
        result_data = []
        for i, row in enumerate(keyword_results, 1):
            print(f"\n{i}. Document: {row['document_name']}")
            print(f"   Topic: {row['topic']}")
            print(f"   Chunk: {row['chunk_text'][:200]}...")
            print("-" * 100)
            
            result_data.append({
                "Rank": i,
                "Document": row['document_name'],
                "Topic": row['topic'],
                "Relevant to Stock Options?": "❌ No - False Positive"
            })
        
        # Summary table
        print("\n📈 ANALYSIS SUMMARY")
        print("=" * 100)
        df_results = pd.DataFrame(result_data)
        display(df_results)
        
        print("\n🏷️ Analysis:")
        print("  • Keywords found, but NOT about stock options/compensation")
        print("  • Words like 'option' may appear in different contexts (e.g., 'policy option')")
        print("  • This demonstrates why semantic search alone isn't sufficient")
        
    else:
        print("\n✅ ZERO keyword matches found for: stock, option, vesting, equity, compensation")
        print("\n🎯 This confirms: NO documents in the corpus cover stock/compensation topics")
        
        # If vector search was working, show what it would return anyway
        print("\n\n" + "=" * 100)
        print("⚠️ WHAT WOULD VECTOR SEARCH RETURN?")
        print("=" * 100)
        print("""
Even with ZERO relevant content, vector search would still return the 5 "closest" chunks:
  → These would be random HR documents (GDPR, Travel, Security, Performance, etc.)
  → Similarity scores would be LOW (typically < 0.40)
  → These are FORCED matches - the system must return something
  → All results would be FALSE POSITIVES
        """)
        
except Exception as e:
    print(f"\n❌ Error searching: {str(e)}")

# Key findings
print("\n\n" + "=" * 100)
print("🔑 KEY FINDINGS - UNSUPPORTED QUESTION BEHAVIOR")
print("=" * 100)
print("""
1. ❌ ZERO documents about stock options, equity, or compensation in corpus
   → RAG system has no relevant information to retrieve

2. ⚠️ SIMILARITY SCORES are likely LOW (< 0.50)
   → Embedding model recognizes poor semantic match
   → System is attempting to find "closest" match, but nothing is actually close

3. 🎯 RETRIEVED CHUNKS are NOT relevant to stock options
   → May return random HR documents (GDPR, Travel, Security, etc.)
   → These are false positives - the model is forced to return SOMETHING

4. 💡 PRODUCTION HANDLING STRATEGIES:
   a) Set a similarity score THRESHOLD (e.g., reject if score < 0.55)
   b) Implement TOPIC VALIDATION (check if question topic exists in corpus)
   c) Return "I don't have information about this topic" response
   d) Suggest RELATED topics that ARE available
   e) Log unsupported queries to identify content gaps

5. ✅ METADATA FILTERING won't help here
   → No 'Compensation' or 'Stock' topic exists to filter on
   → This is a corpus coverage gap, not a retrieval precision issue
""")

print("\n" + "=" * 100)
print("💡 RECOMMENDATION")
print("=" * 100)
print("""
For production RAG systems:
  ✓ Always validate similarity scores (threshold: 0.50-0.60)
  ✓ Maintain a topic catalog and validate queries against it
  ✓ Gracefully handle out-of-scope questions
  ✓ Monitor unsupported queries to identify document gaps
  ✓ Consider a fallback mechanism (general web search, escalation to human, etc.)
""")

🔍 QUERYING UNSUPPORTED TOPIC
Question: What is the company's stock option vesting schedule?

📚 Available Topics in HR Document Corpus:
  ✓ DEI
  ✓ Exit Process
  ✓ Flexible Work
  ✓ GDPR
  ✓ HR Summary
  ✓ Performance
  ✓ Security
  ✓ Travel

❌ Notice: 'Stock Options', 'Compensation', or 'Equity' topics are NOT in the corpus


SIMULATED SEARCH RESULTS - Best Keyword Matches (Top 5)

Searching for: 'stock', 'option', 'vesting', 'equity', 'compensation'...

✅ ZERO keyword matches found for: stock, option, vesting, equity, compensation

🎯 This confirms: NO documents in the corpus cover stock/compensation topics


⚠️ WHAT WOULD VECTOR SEARCH RETURN?

Even with ZERO relevant content, vector search would still return the 5 "closest" chunks:
  → These would be random HR documents (GDPR, Travel, Security, Performance, etc.)
  → Similarity scores would be LOW (typically < 0.40)
  → These are FORCED matches - the system must return something
  → All results would be FALSE POSITIVES
        


🔑 